# ABT municipal

Junta as tabelas municipais produzidas pelos livros de IBGE, IVS/IDHM e RAIS em uma linha por município.
A chave de junção é o código IBGE de 7 dígitos (**codigo_ibge**).

Os anos de referência permanecem na fonte que os publicou:

- IBGE: **ano** é o ano-base da ABT, 2022. O PIB corrente usa 2023. O valor adicionado setorial usa 2021.
- Urbanização e território: população urbana e rural do Censo 2022 e flags de capital, litoral e região metropolitana, vindas de **02_urbanizacao_territorio.ipynb**.
- IVS/IDHM: níveis de 2010, mais as colunas de variação 2000–2010 de **abt_ivs_2010_var.parquet**. Os indicadores detalhados do Atlas vêm do recorte de 2010 de **ivs_municipio_ano.parquet**.
- RAIS: emprego formal em 2022. **ano** não é repetido, porque colidiria com o ano-base do IBGE.

Execute antes os livros de origem. Os arquivos parquet ficam em **data/** e não são versionados.


## Importações


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


## Configuração

In [2]:
def resolve_repo_root() -> Path:
    """Localiza a raiz do projeto que contém os books e o diretório local de dados."""
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "books").is_dir() and (candidate / "data").is_dir():
            return candidate
    return start


REPO_ROOT = resolve_repo_root()
DATA_DIR = REPO_ROOT / "data"
TRUSTED_DIR = DATA_DIR / "trusted" / "ibge"
ANALYTICS_DIR = DATA_DIR / "analytics"

IBGE_CANDIDATES = [
    ANALYTICS_DIR / "abt_municipios_ibge_basica.parquet",
    ANALYTICS_DIR / "abt_municipios_ibge_painel.parquet",
]
IVS_VARIATION_PATH = TRUSTED_DIR / "abt_ivs_2010_var.parquet"
IVS_YEAR_PATH = TRUSTED_DIR / "ivs_municipio_ano.parquet"
URBAN_PATH = TRUSTED_DIR / "urbanizacao_municipio_censo_2022.parquet"
TERRITORY_PATH = TRUSTED_DIR / "municipio_litoral_rm.parquet"
RAIS_PATH = TRUSTED_DIR / "rais_2022_municipio.parquet"
TERRITORY_ZERO_FILL = [
    "flag_litoranea_ibge",
    "flag_capital_uf",
    "flag_regiao_metropolitana_ibge",
    "flag_regiao_metropolitana_relevante",
    "qtd_municipios_rm",
    "populacao_total_rm",
    "flag_rm_contem_capital",
    "flag_rm_polo_grande",
    "flag_rm_pop_total_grande",
]
OUTPUT_PATH = ANALYTICS_DIR / "abt_municipios_final.parquet"

KEY = "codigo_ibge"
IVS_REFERENCE_YEAR = 2010


def first_existing(paths: list[Path]) -> Path:
    """Devolve o primeiro caminho que existe ou interrompe listando todos os candidatos."""
    for path in paths:
        if path.is_file():
            return path
    listed = "\n".join(f"- {path}" for path in paths)
    raise FileNotFoundError(f"Nenhum dos arquivos esperados existe:\n{listed}")


def require_file(path: Path) -> Path:
    """Interrompe com o caminho do parquet ausente antes de a junção começar."""
    if not path.is_file():
        raise FileNotFoundError(
            f"Arquivo ausente: {path}. Execute o livro de origem que grava esta tabela."
        )
    return path


## Leitura

Cada fonte fica com uma linha por município. **ano** é mantido só na tabela do IBGE.


In [3]:
def normalize_codigo_ibge(series: pd.Series) -> pd.Series:
    """Converte o código do município para uma chave de texto com 7 dígitos."""
    values = series.astype("string").str.replace(r"\.0$", "", regex=True).str.strip()
    return values.str.zfill(7)


def assert_unique_key(df: pd.DataFrame, source: str) -> None:
    """Recusa uma fonte que não tenha uma linha por município."""
    if KEY not in df.columns:
        raise KeyError(f"{source} não tem a coluna {KEY}.")
    duplicated = int(df[KEY].duplicated().sum())
    if duplicated:
        raise ValueError(f"{source} tem {duplicated} valores duplicados de {KEY}.")


def prepare_municipality_table(
    df: pd.DataFrame,
    source: str,
    *,
    drop_year: bool,
) -> pd.DataFrame:
    """Normaliza a chave de junção e, se pedido, remove a coluna de ano específica da fonte."""
    out = df.copy()
    out[KEY] = normalize_codigo_ibge(out[KEY])
    if drop_year and "ano" in out.columns:
        out = out.drop(columns=["ano"])
    assert_unique_key(out, source)
    return out


def rename_overlaps(
    df: pd.DataFrame,
    reserved: set[str],
    suffix: str,
) -> pd.DataFrame:
    """Renomeia colunas que não são a chave e que já existem no lado esquerdo da junção."""
    overlap = [column for column in df.columns if column in reserved and column != KEY]
    if overlap:
        print(f"{suffix}: colunas sobrepostas renomeadas {overlap}")
        df = df.rename(columns={column: f"{column}{suffix}" for column in overlap})
    return df


def filter_ivs_reference_year(df: pd.DataFrame) -> pd.DataFrame:
    """Mantém o ano municipal do Atlas usado pela tabela de variação 2000–2010."""
    if "ano" not in df.columns:
        raise KeyError(f"{IVS_YEAR_PATH.name} não tem a coluna ano.")
    year = pd.to_numeric(df["ano"], errors="coerce")
    out = df.loc[year == IVS_REFERENCE_YEAR].copy()
    if out.empty:
        raise ValueError(
            f"{IVS_YEAR_PATH.name} não tem linhas para {IVS_REFERENCE_YEAR}."
        )
    return out


ibge_path = first_existing(IBGE_CANDIDATES)
ibge = prepare_municipality_table(
    pd.read_parquet(ibge_path),
    ibge_path.name,
    drop_year=False,
)

# Colunas de nível e de variação absoluta são gravadas com log1p na ABT de referência.
LEVEL_LOG1P_COLUMNS = [
    "pib_corrente",
    "pib_corrente_lag_1y",
    "pib_corrente_lag_2y",
    "pib_corrente_lag_3y",
    "pib_corrente_var_abs_1y",
    "pib_corrente_var_abs_2y",
    "pib_corrente_var_abs_3y",
    "pib_corrente_media_movel_3y",
    "pib_corrente_desvio_movel_3y",
    "pib_per_capita_aprox",
    "vab_agropecuaria",
    "vab_industria",
    "vab_servicos",
    "vab_adm_publica",
    "vab_agropecuaria_lag_1y",
    "vab_agropecuaria_lag_2y",
    "vab_agropecuaria_var_abs_1y",
    "vab_agropecuaria_var_abs_2y",
    "vab_industria_lag_1y",
    "vab_industria_lag_2y",
    "vab_industria_var_abs_1y",
    "vab_industria_var_abs_2y",
    "vab_servicos_lag_1y",
    "vab_servicos_lag_2y",
    "vab_servicos_var_abs_1y",
    "vab_servicos_var_abs_2y",
    "vab_adm_publica_lag_1y",
    "vab_adm_publica_lag_2y",
    "vab_adm_publica_var_abs_1y",
    "vab_adm_publica_var_abs_2y",
    "vab_total_setorial",
    "vab_servicos_total",
    "impostos_liquidos_lag_1y",
    "impostos_liquidos_lag_2y",
    "impostos_liquidos_var_abs_1y",
    "impostos_liquidos_var_abs_2y",
]
level_columns = [column for column in LEVEL_LOG1P_COLUMNS if column in ibge.columns]
ibge[level_columns] = np.log1p(ibge[level_columns])

ivs_variation = prepare_municipality_table(
    pd.read_parquet(require_file(IVS_VARIATION_PATH)),
    IVS_VARIATION_PATH.name,
    drop_year=True,
)

ivs_year = filter_ivs_reference_year(pd.read_parquet(require_file(IVS_YEAR_PATH)))
ivs_year = prepare_municipality_table(
    ivs_year,
    f"{IVS_YEAR_PATH.name}:{IVS_REFERENCE_YEAR}",
    drop_year=True,
)
extra_ivs_columns = [
    column for column in ivs_year.columns if column not in ivs_variation.columns
]
ivs_indicators = ivs_year[[KEY, *extra_ivs_columns]]

rais = prepare_municipality_table(
    pd.read_parquet(require_file(RAIS_PATH)),
    RAIS_PATH.name,
    drop_year=True,
)

urban = prepare_municipality_table(
    pd.read_parquet(require_file(URBAN_PATH)),
    URBAN_PATH.name,
    drop_year=True,
)
urban = urban.drop(
    columns=[column for column in ["municipio_sidra", "populacao_total_censo_2022"] if column in urban.columns]
)

territory = prepare_municipality_table(
    pd.read_parquet(require_file(TERRITORY_PATH)),
    TERRITORY_PATH.name,
    drop_year=True,
)

print(f"IBGE: {ibge_path.name} {ibge.shape}")
print(f"Urbanização: {urban.shape}")
print(f"Flags territoriais: {territory.shape}")
print(f"Variação do IVS: {ivs_variation.shape}")
print(f"IVS {IVS_REFERENCE_YEAR}, indicadores fora da tabela de variação: {ivs_indicators.shape}")
print(f"RAIS: {rais.shape}")


IBGE: abt_municipios_ibge_basica.parquet (5570, 127)
Urbanization: (5570, 5)
Territory flags: (5571, 10)
IVS variation: (5565, 89)
IVS 2010 indicators not in the variation table: (5565, 89)
RAIS: (5571, 13)


## Junção

A lista de municípios do IBGE é o eixo. IVS/IDHM e RAIS entram por junção à esquerda, então um município ausente em um livro posterior permanece na ABT com medidas nulas.

PIB, valor adicionado setorial, impostos líquidos e PIB per capita aproximado, inclusive defasagens e variações absolutas, são gravados com **log1p**, no mesmo padrão de **df_final_abt.parquet**.


In [4]:
reserved = set(ibge.columns)
urban = rename_overlaps(urban, reserved, "_urb")
reserved.update(urban.columns)

territory = rename_overlaps(territory, reserved, "_territorio")
reserved.update(territory.columns)

ivs_variation = rename_overlaps(ivs_variation, reserved, "_ivs")
reserved.update(ivs_variation.columns)

ivs_indicators = rename_overlaps(ivs_indicators, reserved, "_ivs_nivel")
reserved.update(ivs_indicators.columns)

rais = rename_overlaps(rais, reserved, "_rais")

abt_final = (
    ibge.merge(urban, on=KEY, how="left", validate="one_to_one")
    .merge(territory, on=KEY, how="left", validate="one_to_one")
    .merge(ivs_variation, on=KEY, how="left", validate="one_to_one")
    .merge(ivs_indicators, on=KEY, how="left", validate="one_to_one")
    .merge(rais, on=KEY, how="left", validate="one_to_one")
)
for column in TERRITORY_ZERO_FILL:
    if column in abt_final.columns:
        abt_final[column] = abt_final[column].fillna(0)

identity_columns = [
    KEY,
    "ano",
    "municipio",
    "uf",
    "nome_uf",
    "regiao",
    "sigla_regiao",
    "regiao_imediata",
    "regiao_intermediaria",
]
identity_columns = [column for column in identity_columns if column in abt_final.columns]
other_columns = [column for column in abt_final.columns if column not in identity_columns]
abt_final = abt_final[identity_columns + other_columns].sort_values(KEY).reset_index(drop=True)

assert_unique_key(abt_final, OUTPUT_PATH.name)

ANALYTICS_DIR.mkdir(parents=True, exist_ok=True)
abt_final.to_parquet(OUTPUT_PATH, index=False)

print(f"Salvo em {OUTPUT_PATH}")
print(f"formato: {abt_final.shape}")


Saved /Users/lucas.nascimento/city-information-books/data/analytics/abt_municipios_final.parquet
shape: (5570, 328)


## Cobertura da junção

In [5]:
coverage_columns = {
    "urban_population": "populacao_urbana_2022",
    "coastal_flag": "flag_litoranea_ibge",
    "capital_flag": "flag_capital_uf",
    "metropolitan_flag": "flag_regiao_metropolitana_ibge",
    "ivs": "ivs",
    "idhm": "idhm",
    "rais_vinculos": "qtd_vinculos_rais",
}

coverage = []
for source, column in coverage_columns.items():
    if column not in abt_final.columns:
        coverage.append({"source": source, "column": column, "municipalities_with_value": 0})
        continue
    coverage.append(
        {
            "source": source,
            "column": column,
            "municipalities_with_value": int(abt_final[column].notna().sum()),
            "municipalities_missing": int(abt_final[column].isna().sum()),
        }
    )

ibge_keys = set(ibge[KEY])
print("Códigos de urbanização ausentes no IBGE:", len(set(urban[KEY]) - ibge_keys))
print("Códigos de território ausentes no IBGE:", len(set(territory[KEY]) - ibge_keys))
print("Códigos do IVS ausentes no IBGE:", len(set(ivs_variation[KEY]) - ibge_keys))
print("Códigos da RAIS ausentes no IBGE:", len(set(rais[KEY]) - ibge_keys))

pd.DataFrame(coverage)


Urbanization codes absent from IBGE: 0
Territory codes absent from IBGE: 1
IVS codes absent from IBGE: 0
RAIS codes absent from IBGE: 1


,source,column,municipalities_with_value,municipalities_missing
0,urban_population,populacao_urbana_2022,5570,0
1,coastal_flag,flag_litoranea_ibge,5570,0
2,capital_flag,flag_capital_uf,5570,0
3,metropolitan_flag,flag_regiao_metropolitana_ibge,5570,0
4,ivs,ivs,5565,5
5,idhm,idhm,5565,5
6,rais_vinculos,qtd_vinculos_rais,5570,0
